# Video Translation Pipeline - Debug Notebook

This notebook provides an interactive debugging environment for the cloud video translation pipeline.
Each stage of the pipeline is broken into separate cells, allowing you to:

- Inspect intermediate outputs at each stage
- Debug translation quality issues
- Test different configurations
- Understand how each component works

## Pipeline Overview

1. **Setup & Configuration** - Load dependencies and API clients
2. **Input Configuration** - Set video URL and target language
3. **Video Info Retrieval** - Fetch metadata without downloading
4. **Video Download** - Download and optionally trim the video
5. **Audio Extraction** - Extract audio for processing
6. **Audio Separation** - Separate vocals from background music
7. **Speaker Diarization** - Identify different speakers
8. **Voice Sample Extraction** - Extract samples for voice cloning
9. **Transcription** - Convert speech to text
10. **Translation** - Translate text to target language
11. **Speech Synthesis** - Generate translated speech
12. **Audio Mixing** - Mix speech with background
13. **Subtitle Generation** - Create SRT subtitles
14. **Final Merge** - Combine everything into final video

## Cell 1: Setup and Configuration

This cell loads all required dependencies and initializes API clients.

### Dependencies

- **yt-dlp**: Downloads videos from YouTube and 1700+ sites
- **openai**: OpenAI Whisper API for speech-to-text transcription
- **replicate**: Replicate API for:
  - Demucs (audio separation)
  - Speaker diarization
  - Llama (LLM translation)
  - Chatterbox (TTS with voice cloning)
- **deep_translator**: Google Translate fallback for translation
- **httpx**: Async HTTP client for file downloads
- **numpy**: Audio data manipulation

### Environment Variables

Required API keys (loaded from `.env`):
- `OPENAI_API_KEY`: For Whisper transcription
- `REPLICATE_API_TOKEN`: For all Replicate models

Optional configuration:
- `OUTPUT_DIR`: Where to save output files (default: `/tmp/yt-translate`)
- `OXYLABS_PROXY`: Proxy for YouTube downloads (to avoid rate limits)
- `YOUTUBE_COOKIES`: Netscape cookie file for authenticated downloads

In [ ]:
# =============================================================================
# Setup and Configuration
# =============================================================================

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Standard library imports
import asyncio
import os
import re
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Optional

# Third-party imports
import httpx
import numpy as np
import replicate
from replicate.helpers import FileOutput
import yt_dlp
from deep_translator import GoogleTranslator
from openai import OpenAI

# Project imports - language mappings
from languages import (
    SUPPORTED_LANGUAGES,
    GOOGLE_LANG_CODES,
    ISO_639_1_TO_639_2,
    LANG_NAMES,
    get_language_name,
    get_google_code,
    get_iso_639_2_code,
)

# Project imports - configuration constants
from config import (
    GOOGLE_TRANSLATE_CHUNK_SIZE,
    LLM_MAX_SEGMENTS_PER_BATCH,
    LLM_BATCH_OVERLAP,
    LLM_MAX_TOKENS,
    LLM_TEMPERATURE,
    CHATTERBOX_SAMPLE_RATE,
    MIN_VOICE_SAMPLE_DURATION,
    MAX_VOICE_SAMPLE_DURATION,
    SIGNED_URL_EXPIRATION,
    REPLICATE_MODEL_LLAMA,
    REPLICATE_MODEL_CHATTERBOX,
    REPLICATE_MODEL_DEMUCS,
    REPLICATE_MODEL_DIARIZATION,
)

# Project imports - audio utilities
from utils.audio_helpers import read_wav_file, write_wav_file, resample_audio

# =============================================================================
# Helper Functions from cloud_translate.py
# =============================================================================

def parse_timestamp(ts: str) -> float:
    """
    Convert a timestamp string to float seconds.
    
    Handles format like "0:00:00.497812" (H:MM:SS.microseconds)
    or "1:30:45.5" (H:MM:SS.fraction).
    
    Args:
        ts: Timestamp string in format "H:MM:SS.fraction" or "H:MM:SS"
    
    Returns:
        Time in seconds as float (e.g., "1:30:45.5" -> 5445.5)
    """
    parts = ts.split(":")
    if len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = float(parts[2])
        return hours * 3600 + minutes * 60 + seconds
    elif len(parts) == 2:
        minutes = int(parts[0])
        seconds = float(parts[1])
        return minutes * 60 + seconds
    else:
        return float(ts)


def format_timestamp_srt(seconds: float) -> str:
    """
    Convert seconds to SRT timestamp format (HH:MM:SS,mmm).
    
    Args:
        seconds: Time in seconds
    
    Returns:
        Timestamp string in SRT format
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


# =============================================================================
# Configuration from Environment
# =============================================================================

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
REPLICATE_API_TOKEN = os.getenv("REPLICATE_API_TOKEN")

# Output directory
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "/tmp/yt-translate"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Proxy configuration (optional)
OXYLABS_PROXY = os.getenv("OXYLABS_PROXY")
YOUTUBE_COOKIES = os.getenv("YOUTUBE_COOKIES")

# =============================================================================
# Initialize API Clients
# =============================================================================

# OpenAI client for Whisper API
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# Replicate client is initialized via environment variable automatically
# It reads REPLICATE_API_TOKEN from the environment

# =============================================================================
# yt-dlp Configuration Helper
# =============================================================================

def get_ytdlp_opts(extra_opts: dict = None) -> dict:
    """Get yt-dlp options with optional proxy and cookie support."""
    opts = {'quiet': True, 'no_warnings': True}
    if OXYLABS_PROXY:
        opts['proxy'] = OXYLABS_PROXY
    if YOUTUBE_COOKIES:
        # Write cookies to temp file for yt-dlp
        cookie_file = OUTPUT_DIR / "youtube_cookies.txt"
        cookie_file.write_text(YOUTUBE_COOKIES)
        opts['cookiefile'] = str(cookie_file)
    if extra_opts:
        opts.update(extra_opts)
    return opts


# =============================================================================
# Display Configuration Status
# =============================================================================

print("=" * 60)
print("CONFIGURATION STATUS")
print("=" * 60)
print()
print(f"Output Directory: {OUTPUT_DIR}")
print(f"  - Exists: {OUTPUT_DIR.exists()}")
print()
print("API Keys:")
print(f"  - OPENAI_API_KEY: {'Set (' + OPENAI_API_KEY[:8] + '...)' if OPENAI_API_KEY else 'NOT SET'}")
print(f"  - REPLICATE_API_TOKEN: {'Set (' + REPLICATE_API_TOKEN[:8] + '...)' if REPLICATE_API_TOKEN else 'NOT SET'}")
print()
print("API Clients:")
print(f"  - OpenAI Client: {'Initialized' if openai_client else 'NOT INITIALIZED (missing API key)'}")
print(f"  - Replicate: {'Available' if REPLICATE_API_TOKEN else 'NOT AVAILABLE (missing API token)'}")
print()
print("Optional Configuration:")
print(f"  - Proxy (OXYLABS_PROXY): {'Configured' if OXYLABS_PROXY else 'Not configured'}")
print(f"  - YouTube Cookies: {'Configured' if YOUTUBE_COOKIES else 'Not configured'}")
print()
print("Replicate Models:")
print(f"  - LLM (Llama): {REPLICATE_MODEL_LLAMA}")
print(f"  - TTS (Chatterbox): {REPLICATE_MODEL_CHATTERBOX.split(':')[0]}")
print(f"  - Audio Separation (Demucs): {REPLICATE_MODEL_DEMUCS.split(':')[0]}")
print(f"  - Speaker Diarization: {REPLICATE_MODEL_DIARIZATION.split(':')[0]}")
print()
print("Pipeline Constants:")
print(f"  - Chatterbox Sample Rate: {CHATTERBOX_SAMPLE_RATE} Hz")
print(f"  - Min Voice Sample Duration: {MIN_VOICE_SAMPLE_DURATION} seconds")
print(f"  - Max Voice Sample Duration: {MAX_VOICE_SAMPLE_DURATION} seconds")
print(f"  - LLM Max Segments per Batch: {LLM_MAX_SEGMENTS_PER_BATCH}")
print(f"  - LLM Batch Overlap: {LLM_BATCH_OVERLAP}")
print(f"  - LLM Temperature: {LLM_TEMPERATURE}")
print()
print(f"Supported Languages: {len(SUPPORTED_LANGUAGES)}")
print(f"  {', '.join(SUPPORTED_LANGUAGES.values())}")
print()
print("=" * 60)
print("Setup complete! Proceed to the next cell to configure input.")
print("=" * 60)

## Cell 2: Input Configuration

This cell defines the input parameters for the pipeline. Modify these values to test different videos and configurations.

### Input Variables

| Variable | Description | Example |
|----------|-------------|--------|
| `VIDEO_URL` | URL of the video to translate | YouTube, Vimeo, Rumble, and 1700+ sites supported |
| `TARGET_LANGUAGE` | 2-letter ISO 639-1 language code | `"es"` (Spanish), `"ja"` (Japanese), `"fr"` (French) |
| `DURATION_LIMIT` | Max video length in seconds (None = full video) | `60` for 1 minute, `None` for no limit |
| `USE_MULTI_SPEAKER` | Enable per-speaker voice cloning | `True` for multiple speakers, `False` for single voice |

### Supported URL Formats

- **YouTube**: `https://www.youtube.com/watch?v=VIDEO_ID` or `https://youtu.be/VIDEO_ID`
- **Vimeo**: `https://vimeo.com/VIDEO_ID`
- **Rumble**: `https://rumble.com/VIDEO_ID.html`
- **Many more**: yt-dlp supports 1700+ sites ([full list](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md))

### Supported Language Codes

| Code | Language | Code | Language | Code | Language |
|------|----------|------|----------|------|----------|
| `ar` | Arabic | `he` | Hebrew | `no` | Norwegian |
| `zh` | Chinese | `hi` | Hindi | `pl` | Polish |
| `da` | Danish | `it` | Italian | `pt` | Portuguese |
| `nl` | Dutch | `ja` | Japanese | `ru` | Russian |
| `en` | English | `ko` | Korean | `es` | Spanish |
| `fi` | Finnish | `ms` | Malay | `sw` | Swahili |
| `fr` | French | `sv` | Swedish | `tr` | Turkish |
| `de` | German | `el` | Greek | | |

### Multi-Speaker Mode

- **`USE_MULTI_SPEAKER = True`**: Uses speaker diarization to identify different speakers, then clones each speaker's voice for synthesis. Best for interviews, podcasts, and multi-person content.
- **`USE_MULTI_SPEAKER = False`**: Uses a single voice for all speech synthesis. Faster and simpler, best for single-speaker content like tutorials or narration.

In [ ]:
# =============================================================================
# Input Configuration - Modify these values for your video
# =============================================================================

# Video URL to translate
# Supports: YouTube, Vimeo, Rumble, and 1700+ sites via yt-dlp
VIDEO_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # Example: Rick Astley

# Target language (2-letter ISO 639-1 code)
# Examples: "es" (Spanish), "ja" (Japanese), "fr" (French), "de" (German)
TARGET_LANGUAGE = "es"  # Spanish

# Duration limit in seconds (None = process full video)
# Useful for testing with long videos - set to 60 for first minute only
DURATION_LIMIT: int | None = 60  # Process only first 60 seconds

# Multi-speaker mode toggle
# True = Use speaker diarization and per-speaker voice cloning (better for interviews/podcasts)
# False = Single voice synthesis (faster, simpler, better for single-speaker content)
USE_MULTI_SPEAKER = True

# =============================================================================
# Display Configuration Summary
# =============================================================================

print("=" * 60)
print("INPUT CONFIGURATION")
print("=" * 60)
print()
print(f"Video URL: {VIDEO_URL}")
print(f"Target Language: {TARGET_LANGUAGE} ({get_language_name(TARGET_LANGUAGE)})")
print(f"Duration Limit: {DURATION_LIMIT} seconds" if DURATION_LIMIT else "Duration Limit: None (full video)")
print(f"Multi-Speaker Mode: {'Enabled' if USE_MULTI_SPEAKER else 'Disabled'}")
print()

# Validate target language
if TARGET_LANGUAGE not in SUPPORTED_LANGUAGES:
    print(f"WARNING: '{TARGET_LANGUAGE}' is not in supported languages!")
    print(f"   Supported: {list(SUPPORTED_LANGUAGES.keys())}")
else:
    print(f"Target language '{TARGET_LANGUAGE}' is supported")

print()
print("=" * 60)
print("Configuration complete! Proceed to video info retrieval.")
print("=" * 60)

## Cell 3: Video Info Retrieval

This cell fetches video metadata **without downloading** the actual video file.

### How it works

yt-dlp's `extract_info()` with `download=False` makes an API call to the video platform to retrieve metadata:

- **Title**: Video title as shown on the platform
- **Duration**: Length in seconds (needed for price calculation)
- **Video ID**: Platform-specific identifier (e.g., YouTube's 11-character ID)
- **Thumbnail URL**: Preview image URL

### Price Calculation

The estimated price is calculated using the formula:

```
price = ceil(duration_seconds / 60) * PRICE_PER_MINUTE
```

- Rounded up to the nearest minute
- Minimum charge is 1 minute
- Default rate: $0.50 per minute (configurable via `PRICE_PER_MINUTE_CENTS`)

### Error Handling

Common errors include:
- **Invalid URL**: Unsupported platform or malformed URL
- **Private/Restricted**: Video requires authentication or is geo-blocked
- **Removed/Unavailable**: Video no longer exists

In [ ]:
# =============================================================================
# Video Info Retrieval - Fetch metadata without downloading
# =============================================================================

# Price per minute in cents (configurable via environment variable)
PRICE_PER_MINUTE_CENTS = int(os.getenv("PRICE_PER_MINUTE_CENTS", "50"))  # $0.50 default


def calculate_price_cents(duration_seconds: int) -> int:
    """
    Calculate price in cents based on video duration.
    
    Price is calculated per minute, rounded up.
    Minimum price is 1 minute.
    """
    minutes = max(1, (duration_seconds + 59) // 60)  # Round up to nearest minute
    return minutes * PRICE_PER_MINUTE_CENTS


def format_duration(seconds: int) -> str:
    """Format duration in seconds to human-readable string (HH:MM:SS or MM:SS)."""
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


# =============================================================================
# Extract Video Info
# =============================================================================

video_info = None  # Will hold the extracted info for later cells

print("=" * 60)
print("VIDEO INFO RETRIEVAL")
print("=" * 60)
print()
print(f"Fetching info for: {VIDEO_URL}")
print()

try:
    with yt_dlp.YoutubeDL(get_ytdlp_opts()) as ydl:
        info = ydl.extract_info(VIDEO_URL, download=False)
        
        if info is None:
            raise ValueError("Could not extract video information from the provided URL")
        
        # Extract metadata
        title = info.get('title', 'Unknown Title')
        duration = info.get('duration')
        video_id = info.get('id', 'unknown')
        
        if duration is None:
            raise ValueError("Could not determine video duration. The URL may not point to a valid video.")
        
        # Get thumbnail (yt-dlp provides various thumbnail options)
        thumbnail = info.get('thumbnail')
        if not thumbnail:
            thumbnails = info.get('thumbnails', [])
            if thumbnails:
                thumbnail = thumbnails[-1].get('url')
        
        # Calculate price
        price_cents = calculate_price_cents(duration)
        
        # Store for later cells
        video_info = {
            'title': title,
            'duration': duration,
            'video_id': video_id,
            'thumbnail': thumbnail,
            'price_cents': price_cents,
            'uploader': info.get('uploader', 'Unknown'),
            'extractor': info.get('extractor', 'Unknown'),
        }
        
        # Display results
        print("Video Metadata")
        print("-" * 40)
        print(f"Title:      {title}")
        print(f"Video ID:   {video_id}")
        print(f"Duration:   {format_duration(duration)} ({duration} seconds)")
        print(f"Uploader:   {video_info['uploader']}")
        print(f"Platform:   {video_info['extractor']}")
        print()
        print(f"Thumbnail URL:")
        print(f"  {thumbnail}")
        print()
        print("Price Estimation")
        print("-" * 40)
        minutes = max(1, (duration + 59) // 60)
        print(f"Billable minutes: {minutes} (rounded up from {duration/60:.1f} min)")
        print(f"Rate: ${PRICE_PER_MINUTE_CENTS/100:.2f} per minute")
        print(f"Estimated price: ${price_cents/100:.2f} ({price_cents} cents)")
        print()
        
        # Note about duration limit
        if DURATION_LIMIT and duration > DURATION_LIMIT:
            limited_price_cents = calculate_price_cents(DURATION_LIMIT)
            print(f"With DURATION_LIMIT={DURATION_LIMIT}s: ${limited_price_cents/100:.2f}")
            print(f"  (Processing only first {format_duration(DURATION_LIMIT)} of {format_duration(duration)})")
        
        print()
        print("=" * 60)
        print("Video info retrieved successfully! Proceed to download.")
        print("=" * 60)

except yt_dlp.utils.DownloadError as e:
    print(f"ERROR: Invalid video URL or video not accessible")
    print(f"  Details: {str(e)}")
    print()
    print("Common causes:")
    print("  - URL is malformed or unsupported")
    print("  - Video is private or age-restricted")
    print("  - Video has been removed")
    print("  - Geographic restrictions apply")
    print()
    print("Try: Double-check the URL or use YouTube cookies for restricted videos.")

except Exception as e:
    print(f"ERROR: Failed to extract video information")
    print(f"  Exception type: {type(e).__name__}")
    print(f"  Details: {str(e)}")

## Cell 4: Video Download

This cell downloads the video file using yt-dlp and optionally trims it if `DURATION_LIMIT` is set.

### Download Process

1. **Format Selection**: yt-dlp selects `bestvideo+bestaudio/best` - the highest quality video and audio streams
2. **Merge Output**: Streams are merged into a single MP4 container using ffmpeg
3. **Duration Trimming**: If `DURATION_LIMIT` is set, ffmpeg trims the video using stream copy (fast, no re-encoding)

### Format Selection Details

The `bestvideo+bestaudio/best` format string means:
- Try to get the best video and best audio as separate streams, then merge them
- If that fails (e.g., audio-only or video-only), fall back to the single best stream

### Duration Trimming

When `DURATION_LIMIT` is set:
```bash
ffmpeg -i input.mp4 -t {DURATION_LIMIT} -c:v copy -c:a copy -y output.mp4
```

- `-t {DURATION_LIMIT}`: Stop writing output after this many seconds
- `-c:v copy -c:a copy`: Stream copy (no re-encoding) - very fast
- The original file is replaced with the trimmed version

### Output

- **video_path**: Path to the downloaded (and optionally trimmed) video file
- **File size**: Displayed in human-readable format (MB/GB)

In [ ]:
# =============================================================================
# Video Download - Download and optionally trim the video
# =============================================================================

def format_file_size(size_bytes: int) -> str:
    """Format file size in human-readable format."""
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.1f} KB"
    elif size_bytes < 1024 * 1024 * 1024:
        return f"{size_bytes / (1024 * 1024):.1f} MB"
    else:
        return f"{size_bytes / (1024 * 1024 * 1024):.2f} GB"


# =============================================================================
# Download Video
# =============================================================================

video_path = None  # Will hold the path to the downloaded video

print("=" * 60)
print("VIDEO DOWNLOAD")
print("=" * 60)
print()

# Check prerequisite from previous cell
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
else:
    try:
        video_id = video_info['video_id']
        title = video_info['title']
        duration = video_info['duration']
        
        # Create job directory for this video
        job_dir = OUTPUT_DIR / video_id
        job_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Downloading: {title}")
        print(f"Video ID: {video_id}")
        print(f"Output directory: {job_dir}")
        print()
        
        # Configure yt-dlp for download
        ydl_opts = get_ytdlp_opts({
            'format': 'bestvideo+bestaudio/best',
            'outtmpl': str(job_dir / f"{video_id}.%(ext)s"),
            'merge_output_format': 'mp4',
            'quiet': False,  # Show progress for debugging
            'no_warnings': False,
        })
        
        # Download the video
        print("Downloading video...")
        print("-" * 40)
        download_start = time.time()
        
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([VIDEO_URL])
        
        download_time = time.time() - download_start
        print("-" * 40)
        print(f"Download completed in {download_time:.1f} seconds")
        print()
        
        # Find the downloaded video file
        video_path = None
        for ext in ['mp4', 'mkv', 'webm']:
            p = job_dir / f"{video_id}.{ext}"
            if p.exists():
                video_path = p
                break
        
        if video_path is None:
            raise FileNotFoundError(f"Could not find downloaded video in {job_dir}")
        
        original_size = video_path.stat().st_size
        print(f"Downloaded file: {video_path.name}")
        print(f"Original size: {format_file_size(original_size)}")
        print()
        
        # Apply duration limit if specified
        if DURATION_LIMIT is not None and duration > DURATION_LIMIT:
            print(f"Applying duration limit: {DURATION_LIMIT} seconds")
            print(f"  (Trimming from {format_duration(duration)} to {format_duration(DURATION_LIMIT)})")
            print()
            
            trimmed_path = job_dir / f"{video_id}_trimmed.mp4"
            
            # Trim using ffmpeg with stream copy (fast, no re-encoding)
            trim_cmd = [
                'ffmpeg', '-i', str(video_path),
                '-t', str(DURATION_LIMIT),
                '-c:v', 'copy', '-c:a', 'copy',
                '-y', str(trimmed_path)
            ]
            
            print("Running ffmpeg trim...")
            trim_start = time.time()
            result = subprocess.run(trim_cmd, capture_output=True, text=True)
            trim_time = time.time() - trim_start
            
            if result.returncode != 0:
                print(f"ERROR: ffmpeg trim failed")
                print(f"  stderr: {result.stderr}")
            else:
                # Replace original with trimmed version
                video_path.unlink()  # Delete original
                # Rename trimmed to standard name
                final_path = job_dir / f"{video_id}.mp4"
                trimmed_path.rename(final_path)
                video_path = final_path
                
                trimmed_size = video_path.stat().st_size
                print(f"Trim completed in {trim_time:.1f} seconds")
                print(f"Trimmed size: {format_file_size(trimmed_size)}")
                print(f"Size reduction: {(1 - trimmed_size/original_size)*100:.1f}%")
        
        print()
        print("Download Summary")
        print("-" * 40)
        print(f"Video path: {video_path}")
        print(f"Final size: {format_file_size(video_path.stat().st_size)}")
        
        # Update video_info with actual processed duration
        actual_duration = min(duration, DURATION_LIMIT) if DURATION_LIMIT else duration
        video_info['actual_duration'] = actual_duration
        video_info['video_path'] = video_path
        print(f"Actual duration: {format_duration(actual_duration)} ({actual_duration} seconds)")
        
        print()
        print("=" * 60)
        print("Video downloaded successfully! Proceed to audio extraction.")
        print("=" * 60)
        
    except subprocess.CalledProcessError as e:
        print(f"ERROR: ffmpeg command failed")
        print(f"  Command: {' '.join(e.cmd)}")
        print(f"  Return code: {e.returncode}")
        if e.stderr:
            print(f"  stderr: {e.stderr}")
    
    except yt_dlp.utils.DownloadError as e:
        print(f"ERROR: Video download failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Network connectivity issues")
        print("  - Video is age-restricted (try using cookies)")
        print("  - Rate limiting (try using a proxy)")
        print("  - DRM-protected content")
    
    except Exception as e:
        print(f"ERROR: Unexpected error during download")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")

## Cell 5: Audio Extraction

This cell extracts audio from the downloaded video file for processing by the speech-to-text pipeline.

### Why 16kHz Mono PCM?

OpenAI's Whisper API works best with audio that matches its training data format:

- **16kHz Sample Rate**: Whisper was trained on 16kHz audio. Higher rates waste bandwidth without improving accuracy.
- **Mono (1 channel)**: Speech recognition doesn't benefit from stereo. Combining channels reduces file size by 50%.
- **PCM 16-bit**: Uncompressed format with no quality loss. The `.wav` extension signals this to downstream tools.

### ffmpeg Command Breakdown

```bash
ffmpeg -i video.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 -y audio.wav
```

| Flag | Description |
|------|-------------|
| `-i video.mp4` | Input video file |
| `-vn` | No video output (audio only) |
| `-acodec pcm_s16le` | PCM 16-bit signed little-endian codec |
| `-ar 16000` | Set audio sample rate to 16kHz |
| `-ac 1` | Convert to mono (1 audio channel) |
| `-y` | Overwrite output file if exists |

### Output

- **audio_path**: Path to the extracted `.wav` file
- **Audio duration**: Should match video duration (or `DURATION_LIMIT` if set)
- **File size**: Approximately `duration_seconds * 32000` bytes (16000 Hz × 2 bytes × mono)

In [ ]:
# =============================================================================
# Audio Extraction - Extract audio from video for transcription
# =============================================================================

# Audio extraction parameters
WHISPER_SAMPLE_RATE = 16000  # 16kHz for Whisper API


def get_audio_duration(audio_path: Path) -> float:
    """
    Get the duration of an audio file using ffprobe.
    
    Args:
        audio_path: Path to the audio file
    
    Returns:
        Duration in seconds
    """
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(audio_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip():
        return float(result.stdout.strip())
    return 0.0


# =============================================================================
# Extract Audio from Video
# =============================================================================

audio_path = None  # Will hold the path to the extracted audio

print("=" * 60)
print("AUDIO EXTRACTION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('video_path') is None:
    print("ERROR: video_path not available. Run the Video Download cell first.")
else:
    try:
        video_path = video_info['video_path']
        video_id = video_info['video_id']
        job_dir = video_path.parent
        
        # Define output audio path
        audio_path = job_dir / "audio.wav"
        
        print(f"Source video: {video_path}")
        print(f"Output audio: {audio_path}")
        print(f"Target format: {WHISPER_SAMPLE_RATE} Hz, mono, PCM 16-bit")
        print()
        
        # Build ffmpeg command for audio extraction
        ffmpeg_cmd = [
            'ffmpeg',
            '-i', str(video_path),        # Input video
            '-vn',                         # No video output
            '-acodec', 'pcm_s16le',       # PCM 16-bit signed little-endian
            '-ar', str(WHISPER_SAMPLE_RATE),  # 16kHz sample rate
            '-ac', '1',                    # Mono (1 channel)
            '-y',                          # Overwrite output file
            str(audio_path)
        ]
        
        print("Running ffmpeg...")
        print(f"  Command: {' '.join(ffmpeg_cmd)}")
        print()
        
        extract_start = time.time()
        result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
        extract_time = time.time() - extract_start
        
        if result.returncode != 0:
            print(f"ERROR: ffmpeg audio extraction failed")
            print(f"  Return code: {result.returncode}")
            print(f"  stderr: {result.stderr}")
            audio_path = None
        else:
            # Verify the output file exists
            if not audio_path.exists():
                raise FileNotFoundError(f"Audio extraction failed: {audio_path} not created")
            
            # Get file info
            audio_size = audio_path.stat().st_size
            audio_duration = get_audio_duration(audio_path)
            
            # Store in video_info for later cells
            video_info['audio_path'] = audio_path
            video_info['audio_duration'] = audio_duration
            
            # Display results
            print("Audio Extraction Summary")
            print("-" * 40)
            print(f"Audio path: {audio_path}")
            print(f"File size: {format_file_size(audio_size)}")
            print(f"Duration: {format_duration(int(audio_duration))} ({audio_duration:.2f} seconds)")
            print(f"Sample rate: {WHISPER_SAMPLE_RATE} Hz")
            print(f"Channels: 1 (mono)")
            print(f"Bit depth: 16-bit")
            print(f"Extraction time: {extract_time:.2f} seconds")
            print()
            
            # Calculate expected file size and compare
            expected_size = int(audio_duration * WHISPER_SAMPLE_RATE * 2)  # 2 bytes per sample
            expected_size += 44  # WAV header
            print(f"Expected size (approx): {format_file_size(expected_size)}")
            print(f"Actual size: {format_file_size(audio_size)}")
            
            print()
            print("=" * 60)
            print("Audio extracted successfully! Proceed to audio separation.")
            print("=" * 60)
    
    except subprocess.CalledProcessError as e:
        print(f"ERROR: ffmpeg command failed")
        print(f"  Return code: {e.returncode}")
        if e.stderr:
            print(f"  stderr: {e.stderr}")
    
    except FileNotFoundError as e:
        print(f"ERROR: {str(e)}")
    
    except Exception as e:
        print(f"ERROR: Unexpected error during audio extraction")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")

## Cell 6: Audio Separation

This cell separates vocals from background music using Replicate's **Demucs** API. This improves transcription quality by removing non-speech audio.

### Why Separate Audio?

Background music, sound effects, and ambient noise can significantly degrade speech-to-text accuracy. Demucs is a state-of-the-art audio source separation model that isolates:

- **Vocals**: Human speech and singing
- **Drums**: Percussion elements
- **Bass**: Low-frequency instruments
- **Other**: Everything else (background music, effects)

For our pipeline, we keep **vocals** (for transcription) and **other** (as background to mix back in later).

### Demucs API Details

The Replicate model `cjwbw/demucs` accepts:

| Parameter | Description |
|-----------|-------------|
| `audio` | Base64 data URI of the audio file |
| `output_format` | Output format (`wav` recommended for quality) |

Returns a dictionary with URLs for each stem: `vocals`, `drums`, `bass`, `other`.

### Output Files

- **vocals.wav**: Isolated speech for transcription
- **background.wav**: Background audio (from "other" stem) for mixing back later

### Timing

API call timing depends on audio length and server load. Typical processing:
- 60 seconds audio: ~20-30 seconds
- 5 minutes audio: ~1-2 minutes

In [ ]:
# =============================================================================
# Audio Separation - Separate vocals from background using Demucs
# =============================================================================

import base64

# =============================================================================
# Audio Separation
# =============================================================================

vocals_path = None  # Will hold the path to isolated vocals
background_path = None  # Will hold the path to background audio

print("=" * 60)
print("AUDIO SEPARATION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('audio_path') is None:
    print("ERROR: audio_path not available. Run the Audio Extraction cell first.")
else:
    try:
        audio_path = video_info['audio_path']
        job_dir = audio_path.parent
        
        print(f"Source audio: {audio_path}")
        print(f"Model: {REPLICATE_MODEL_DEMUCS.split(':')[0]}")
        print()
        
        # Convert audio to base64 data URI for Replicate
        print("Encoding audio as base64 data URI...")
        with open(audio_path, "rb") as f:
            audio_data = base64.b64encode(f.read()).decode("utf-8")
        
        # Determine mime type based on file extension
        suffix = audio_path.suffix.lower()
        mime_type = "audio/wav" if suffix == ".wav" else "audio/mpeg"
        data_uri = f"data:{mime_type};base64,{audio_data}"
        
        print(f"  - Audio size: {format_file_size(len(audio_data))} (base64)")
        print(f"  - MIME type: {mime_type}")
        print()
        
        # Call Replicate Demucs API
        print("Calling Replicate Demucs API...")
        print("-" * 40)
        
        api_start = time.time()
        output = replicate.run(
            REPLICATE_MODEL_DEMUCS,
            input={
                "audio": data_uri,
                "output_format": "wav",
            }
        )
        api_time = time.time() - api_start
        
        print("-" * 40)
        print(f"API call completed in {api_time:.1f} seconds")
        print()
        
        # Define output paths
        vocals_path = job_dir / "vocals.wav"
        background_path = job_dir / "background.wav"
        
        # Handle FileOutput objects from Replicate v1.0+
        vocals_url = output["vocals"].url if isinstance(output["vocals"], FileOutput) else output["vocals"]
        other_url = output["other"].url if isinstance(output["other"], FileOutput) else output["other"]
        
        print("Downloading separated stems...")
        
        # Download vocals
        print(f"  - Downloading vocals...")
        download_start = time.time()
        
        # Use synchronous httpx for Jupyter compatibility
        with httpx.Client(timeout=None) as client:
            vocals_response = client.get(vocals_url)
            vocals_path.write_bytes(vocals_response.content)
            
            print(f"  - Downloading background (other stem)...")
            other_response = client.get(other_url)
            background_path.write_bytes(other_response.content)
        
        download_time = time.time() - download_start
        print(f"  - Download completed in {download_time:.1f} seconds")
        print()
        
        # Get file info
        vocals_size = vocals_path.stat().st_size
        background_size = background_path.stat().st_size
        
        # Store in video_info for later cells
        video_info['vocals_path'] = vocals_path
        video_info['background_path'] = background_path
        
        # Display results
        print("Audio Separation Summary")
        print("-" * 40)
        print(f"Vocals path: {vocals_path}")
        print(f"  - Size: {format_file_size(vocals_size)}")
        print()
        print(f"Background path: {background_path}")
        print(f"  - Size: {format_file_size(background_size)}")
        print()
        print("Timing")
        print("-" * 40)
        print(f"API call: {api_time:.1f} seconds")
        print(f"Download: {download_time:.1f} seconds")
        print(f"Total: {api_time + download_time:.1f} seconds")
        
        print()
        print("=" * 60)
        print("Audio separated successfully! Proceed to speaker diarization.")
        print("=" * 60)
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate API call failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Model unavailable")
    
    except httpx.HTTPError as e:
        print(f"ERROR: Failed to download separated audio")
        print(f"  Details: {str(e)}")
    
    except Exception as e:
        print(f"ERROR: Unexpected error during audio separation")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")
        print()
        print("Fallback: Using original audio as vocals (no separation)")
        vocals_path = audio_path
        background_path = audio_path
        video_info['vocals_path'] = vocals_path
        video_info['background_path'] = background_path

## Cell 7: Speaker Diarization

This cell identifies different speakers in the audio using Replicate's **speaker-diarization** API. This is essential for multi-speaker content like interviews, podcasts, or panel discussions.

### What is Speaker Diarization?

Speaker diarization answers the question "who spoke when?" by segmenting audio into time ranges and assigning speaker labels. The model:

1. **Detects speech regions** - Identifies when someone is speaking vs. silence
2. **Clusters speakers** - Groups similar voice characteristics together
3. **Assigns labels** - Tags each segment with a speaker ID (SPEAKER_00, SPEAKER_01, etc.)

### Replicate API Details

The model `meronym/speaker-diarization` requires:

| Parameter | Description |
|-----------|-------------|
| `audio` | URL to the audio file (uploaded via `replicate.files.create()`) |

Returns a JSON object with segments:
```json
{
  "segments": [
    {"speaker": "A", "start": "0:00:00.5", "end": "0:00:05.2"},
    {"speaker": "B", "start": "0:00:05.5", "end": "0:00:10.1"},
    ...
  ]
}
```

### Speaker Label Format

The API returns single-letter labels (A, B, C...) which we convert to the standard format:
- `A` → `SPEAKER_00`
- `B` → `SPEAKER_01`
- etc.

### Output

- **diarization_segments**: List of segments with speaker labels and timestamps
- **Number of speakers detected**: Total unique speakers found
- **Segment table**: Formatted display of all segments

### Timing

Processing time depends on audio length:
- 60 seconds: ~10-20 seconds
- 5 minutes: ~30-60 seconds

In [ ]:
# =============================================================================
# Speaker Diarization - Identify different speakers in the audio
# =============================================================================

import json as json_module  # Avoid conflict with potential variable names

# =============================================================================
# Speaker Diarization
# =============================================================================

diarization_segments = []  # Will hold the parsed diarization segments

print("=" * 60)
print("SPEAKER DIARIZATION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('vocals_path') is None:
    print("ERROR: vocals_path not available. Run the Audio Separation cell first.")
else:
    try:
        vocals_path = video_info['vocals_path']
        job_dir = vocals_path.parent
        
        print(f"Source audio: {vocals_path}")
        print(f"Model: {REPLICATE_MODEL_DIARIZATION.split(':')[0]}")
        print()
        
        # Upload vocals file to Replicate for API access
        # The diarization model requires a URL, not base64 data URI
        print("Uploading vocals to Replicate...")
        upload_start = time.time()
        
        uploaded_file = replicate.files.create(file=vocals_path)
        vocals_url = uploaded_file.urls['get']
        
        upload_time = time.time() - upload_start
        print(f"  - Upload completed in {upload_time:.1f} seconds")
        print(f"  - File URL: {vocals_url[:80]}...")
        print()
        
        # Call Replicate speaker-diarization API
        print("Calling Replicate speaker-diarization API...")
        print("-" * 40)
        
        api_start = time.time()
        output = replicate.run(
            REPLICATE_MODEL_DIARIZATION,
            input={"audio": vocals_url}
        )
        api_time = time.time() - api_start
        
        print("-" * 40)
        print(f"API call completed in {api_time:.1f} seconds")
        print()
        
        # Handle different output types from Replicate v1.0+
        if isinstance(output, FileOutput):
            output_bytes = output.read()
            output = json_module.loads(output_bytes)
        elif isinstance(output, str):
            # Output is a URL pointing to JSON - fetch and parse
            with httpx.Client(timeout=None) as client:
                response = client.get(output)
                output = json_module.loads(response.text)
        
        # Parse output segments and convert speaker labels
        # Model returns: {"segments": [...], "speakers": {...}} or just a list
        segments_list = output.get("segments", []) if isinstance(output, dict) else output
        
        if not segments_list:
            print("WARNING: No speakers detected in audio")
            print("  The audio may not contain speech or the model couldn't detect it.")
            diarization_segments = []
        else:
            # Convert speaker labels from "A", "B" to "SPEAKER_00", "SPEAKER_01" format
            speaker_label_map = {}
            speaker_counter = 0
            
            for seg in segments_list:
                raw_label = seg.get("speaker", "")
                if raw_label not in speaker_label_map:
                    speaker_label_map[raw_label] = f"SPEAKER_{speaker_counter:02d}"
                    speaker_counter += 1
                
                # Parse timestamps - handle both string and numeric formats
                start_val = seg.get("start", "0")
                end_val = seg.get("stop", seg.get("end", "0"))
                
                start_sec = parse_timestamp(str(start_val)) if isinstance(start_val, str) else float(start_val)
                end_sec = parse_timestamp(str(end_val)) if isinstance(end_val, str) else float(end_val)
                
                diarization_segments.append({
                    "speaker": speaker_label_map[raw_label],
                    "start": start_sec,
                    "end": end_sec
                })
            
            # Store in video_info for later cells
            video_info['diarization_segments'] = diarization_segments
            video_info['num_speakers'] = speaker_counter
            
            # Display summary
            print("Diarization Summary")
            print("-" * 40)
            print(f"Number of speakers detected: {speaker_counter}")
            print(f"Total segments: {len(diarization_segments)}")
            print()
            
            # Calculate per-speaker statistics
            speaker_stats = {}
            for seg in diarization_segments:
                spk = seg['speaker']
                duration = seg['end'] - seg['start']
                if spk not in speaker_stats:
                    speaker_stats[spk] = {'count': 0, 'total_duration': 0.0}
                speaker_stats[spk]['count'] += 1
                speaker_stats[spk]['total_duration'] += duration
            
            print("Per-Speaker Statistics")
            print("-" * 40)
            for spk, stats in sorted(speaker_stats.items()):
                print(f"  {spk}: {stats['count']} segments, {stats['total_duration']:.1f}s total")
            print()
            
            # Display segment table
            print("Diarization Segments")
            print("-" * 60)
            print(f"{'#':<4} {'Speaker':<12} {'Start':>10} {'End':>10} {'Duration':>10}")
            print("-" * 60)
            
            for i, seg in enumerate(diarization_segments):
                duration = seg['end'] - seg['start']
                start_str = f"{seg['start']:.2f}s"
                end_str = f"{seg['end']:.2f}s"
                dur_str = f"{duration:.2f}s"
                print(f"{i:<4} {seg['speaker']:<12} {start_str:>10} {end_str:>10} {dur_str:>10}")
            
            print("-" * 60)
        
        print()
        print("Timing")
        print("-" * 40)
        print(f"File upload: {upload_time:.1f} seconds")
        print(f"API call: {api_time:.1f} seconds")
        print(f"Total: {upload_time + api_time:.1f} seconds")
        
        print()
        print("=" * 60)
        print("Speaker diarization complete! Proceed to voice sample extraction.")
        print("=" * 60)
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate API call failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Model unavailable")
        print()
        print("Fallback: Continuing without diarization (single-speaker mode)")
        diarization_segments = []
        video_info['diarization_segments'] = []
        video_info['num_speakers'] = 0
    
    except httpx.HTTPError as e:
        print(f"ERROR: Failed to fetch diarization output")
        print(f"  Details: {str(e)}")
        diarization_segments = []
        video_info['diarization_segments'] = []
        video_info['num_speakers'] = 0
    
    except Exception as e:
        print(f"ERROR: Unexpected error during speaker diarization")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")
        print()
        print("Fallback: Continuing without diarization (single-speaker mode)")
        diarization_segments = []
        video_info['diarization_segments'] = []
        video_info['num_speakers'] = 0

## Cell 8: Voice Sample Extraction

This cell extracts voice samples for each speaker identified during diarization. These samples are used by the Chatterbox TTS model to clone each speaker's voice.

### Why Extract Voice Samples?

Chatterbox (and similar voice cloning models) requires a reference audio sample of each speaker's voice to:

- **Capture voice characteristics**: Pitch, tone, accent, speaking style
- **Enable voice cloning**: Generate synthesized speech that sounds like the original speaker
- **Maintain consistency**: Ensure translated speech sounds natural for each speaker

### Sample Selection Strategy

For each speaker, we select the **longest diarization segment** because:

1. Longer samples provide more acoustic information for voice cloning
2. Longer segments are more likely to be clean speech (not just a grunt or brief interjection)
3. The model performs better with samples in the 3-12 second range

### Duration Requirements

| Parameter | Value | Description |
|-----------|-------|-------------|
| `MIN_VOICE_SAMPLE_DURATION` | 3.0 seconds | Minimum segment length to extract |
| `MAX_VOICE_SAMPLE_DURATION` | 12.0 seconds | Maximum sample duration (longer is capped) |

Speakers with segments shorter than `MIN_VOICE_SAMPLE_DURATION` are skipped, and synthesis will use a fallback voice or the first available speaker's voice.

### ffmpeg Extraction Command

```bash
ffmpeg -y -i vocals.wav -ss {start} -t {duration} {speaker_id}_sample.wav
```

| Flag | Description |
|------|-------------|
| `-y` | Overwrite output file if exists |
| `-i vocals.wav` | Input vocals file |
| `-ss {start}` | Seek to start position (in seconds) |
| `-t {duration}` | Duration to extract (capped at MAX_VOICE_SAMPLE_DURATION) |

### Output

- **Voice sample files**: `{SPEAKER_ID}_sample.wav` for each speaker
- **speaker_voice_urls**: Dictionary mapping speaker IDs to signed URLs for API access
- **Sample durations**: Duration of each extracted sample

In [ ]:
# =============================================================================
# Voice Sample Extraction - Extract samples for each speaker for voice cloning
# =============================================================================

from collections import defaultdict


def extract_speaker_samples(
    audio_path: Path,
    output_dir: Path,
    segments: list[dict],
    target_duration: float = MAX_VOICE_SAMPLE_DURATION,
    min_duration: float = MIN_VOICE_SAMPLE_DURATION
) -> dict[str, Path]:
    """
    Extract voice samples for each speaker using ffmpeg.

    Groups segments by speaker ID, finds the longest segment for each speaker,
    and extracts a voice sample using ffmpeg.

    Args:
        audio_path: Path to the source audio file (vocals)
        output_dir: Directory to save extracted samples
        segments: List of diarization segments with 'speaker', 'start', 'end' keys
        target_duration: Maximum duration for each sample
        min_duration: Minimum segment duration required to extract

    Returns:
        Dict mapping speaker ID to the Path of the extracted sample file.
        Speakers with segments shorter than min_duration are skipped.
    """
    # Group segments by speaker
    speaker_segments: dict[str, list[dict]] = defaultdict(list)
    for seg in segments:
        speaker_id = seg.get("speaker", "SPEAKER_00")
        if speaker_id:  # Skip empty speaker IDs
            speaker_segments[speaker_id].append(seg)

    result: dict[str, Path] = {}

    for speaker_id, segs in speaker_segments.items():
        # Find the longest segment for this speaker
        longest_seg = max(segs, key=lambda s: s["end"] - s["start"])
        seg_duration = longest_seg["end"] - longest_seg["start"]

        # Skip if segment is too short
        if seg_duration < min_duration:
            print(f"  - {speaker_id}: Skipped (longest segment {seg_duration:.1f}s < min {min_duration}s)")
            continue

        # Calculate extraction duration (capped at target_duration)
        extract_duration = min(seg_duration, target_duration)

        # Output file path
        sample_path = output_dir / f"{speaker_id}_sample.wav"

        # Extract using ffmpeg
        cmd = [
            "ffmpeg",
            "-y",  # Overwrite output
            "-i", str(audio_path),
            "-ss", str(longest_seg["start"]),
            "-t", str(extract_duration),
            str(sample_path)
        ]

        try:
            subprocess.run(cmd, check=True, capture_output=True)
            result[speaker_id] = sample_path
            print(f"  - {speaker_id}: Extracted {extract_duration:.1f}s sample from segment at {longest_seg['start']:.1f}s")
        except subprocess.CalledProcessError as e:
            print(f"  - {speaker_id}: ffmpeg failed ({e.returncode})")
            continue

    return result


# =============================================================================
# Extract Voice Samples
# =============================================================================

speaker_samples = {}  # Will hold local paths to voice samples
speaker_voice_urls = {}  # Will hold signed URLs for Replicate API

print("=" * 60)
print("VOICE SAMPLE EXTRACTION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('vocals_path') is None:
    print("ERROR: vocals_path not available. Run the Audio Separation cell first.")
elif video_info.get('diarization_segments') is None:
    print("ERROR: diarization_segments not available. Run the Speaker Diarization cell first.")
else:
    try:
        vocals_path = video_info['vocals_path']
        job_dir = vocals_path.parent
        diarization_segments = video_info['diarization_segments']
        num_speakers = video_info.get('num_speakers', 0)
        
        print(f"Source audio: {vocals_path}")
        print(f"Number of speakers: {num_speakers}")
        print(f"Sample duration: {MIN_VOICE_SAMPLE_DURATION}-{MAX_VOICE_SAMPLE_DURATION} seconds")
        print()
        
        if not diarization_segments:
            print("WARNING: No diarization segments available.")
            print("  Voice sample extraction requires speaker diarization.")
            print("  Pipeline will use single-speaker fallback mode.")
        else:
            # Extract voice samples using ffmpeg
            print("Extracting voice samples...")
            print("-" * 40)
            
            extract_start = time.time()
            speaker_samples = extract_speaker_samples(
                audio_path=vocals_path,
                output_dir=job_dir,
                segments=diarization_segments,
                target_duration=MAX_VOICE_SAMPLE_DURATION,
                min_duration=MIN_VOICE_SAMPLE_DURATION
            )
            extract_time = time.time() - extract_start
            
            print("-" * 40)
            print(f"Extraction completed in {extract_time:.1f} seconds")
            print()
            
            if not speaker_samples:
                print("WARNING: No voice samples extracted.")
                print("  All segments may be too short for extraction.")
                print("  Pipeline will use single-speaker fallback mode.")
            else:
                # Upload samples to Replicate for API access
                print("Uploading voice samples to Replicate...")
                print("-" * 40)
                
                upload_start = time.time()
                for speaker_id, sample_path in speaker_samples.items():
                    try:
                        uploaded_file = replicate.files.create(file=sample_path)
                        speaker_voice_urls[speaker_id] = uploaded_file.urls['get']
                        print(f"  - {speaker_id}: Uploaded successfully")
                    except Exception as e:
                        print(f"  - {speaker_id}: Upload failed ({type(e).__name__}: {str(e)[:50]})")
                
                upload_time = time.time() - upload_start
                print("-" * 40)
                print(f"Upload completed in {upload_time:.1f} seconds")
                print()
                
                # Store in video_info for later cells
                video_info['speaker_samples'] = speaker_samples
                video_info['speaker_voice_urls'] = speaker_voice_urls
                
                # Display summary
                print("Voice Sample Summary")
                print("-" * 60)
                print(f"{'Speaker':<15} {'Duration':>10} {'File Size':>12} {'URL Status':>12}")
                print("-" * 60)
                
                for speaker_id, sample_path in speaker_samples.items():
                    duration = get_audio_duration(sample_path)
                    size = sample_path.stat().st_size
                    has_url = "Uploaded" if speaker_id in speaker_voice_urls else "Failed"
                    print(f"{speaker_id:<15} {duration:>9.1f}s {format_file_size(size):>12} {has_url:>12}")
                
                print("-" * 60)
                print()
                
                # Display speaker_voice_urls dictionary
                print("speaker_voice_urls Dictionary")
                print("-" * 60)
                for speaker_id, url in speaker_voice_urls.items():
                    # Truncate URL for display
                    url_display = url[:60] + "..." if len(url) > 60 else url
                    print(f"  {speaker_id}: {url_display}")
                print()
                
                # Timing summary
                print("Timing")
                print("-" * 40)
                print(f"Sample extraction: {extract_time:.1f} seconds")
                print(f"Sample upload: {upload_time:.1f} seconds")
                print(f"Total: {extract_time + upload_time:.1f} seconds")
        
        print()
        print("=" * 60)
        print("Voice sample extraction complete! Proceed to transcription.")
        print("=" * 60)
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate upload failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Network issues")
    
    except subprocess.CalledProcessError as e:
        print(f"ERROR: ffmpeg extraction failed")
        print(f"  Return code: {e.returncode}")
        if e.stderr:
            print(f"  stderr: {e.stderr.decode()[:200]}")
    
    except Exception as e:
        print(f"ERROR: Unexpected error during voice sample extraction")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")
        print()
        print("Fallback: Pipeline will use single-speaker mode.")
        speaker_samples = {}
        speaker_voice_urls = {}
        video_info['speaker_samples'] = {}
        video_info['speaker_voice_urls'] = {}

## Cell 9: Transcription

This cell transcribes the separated vocals using OpenAI's **Whisper API**. Whisper is a state-of-the-art speech recognition model that converts audio to text with timestamps.

### Whisper API Parameters

The API call uses these parameters:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `model` | `whisper-1` | OpenAI's Whisper model (latest version) |
| `file` | audio file | The vocals.wav file from audio separation |
| `response_format` | `verbose_json` | Returns detailed segment information |
| `timestamp_granularities` | `["segment"]` | Include segment-level timestamps |

### Response Format

The `verbose_json` response includes:

```json
{
  "text": "Full transcription text...",
  "language": "en",
  "duration": 60.0,
  "segments": [
    {
      "id": 0,
      "start": 0.0,
      "end": 5.2,
      "text": "First segment text",
      "tokens": [...],
      "temperature": 0.0,
      ...
    },
    ...
  ]
}
```

### Segment Structure

Each segment contains:

| Field | Type | Description |
|-------|------|-------------|
| `start` | float | Start time in seconds |
| `end` | float | End time in seconds |
| `text` | string | Transcribed text for this segment |
| `tokens` | array | Token IDs (for advanced use) |
| `temperature` | float | Sampling temperature used |

### Output

- **transcription_segments**: List of segments with timestamps and text
- **Detected language**: Whisper auto-detects the source language
- **Segment count**: Total number of segments transcribed
- **Segment table**: Formatted display showing index, timing, and text

### Timing

Processing time depends on audio length and server load:
- 60 seconds audio: ~5-15 seconds
- 5 minutes audio: ~30-60 seconds

In [ ]:
# =============================================================================
# Transcription - Convert speech to text using OpenAI Whisper API
# =============================================================================

# =============================================================================
# Transcription
# =============================================================================

transcription_segments = []  # Will hold the parsed transcription segments

print("=" * 60)
print("TRANSCRIPTION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('vocals_path') is None:
    print("ERROR: vocals_path not available. Run the Audio Separation cell first.")
elif openai_client is None:
    print("ERROR: OpenAI client not initialized. Check OPENAI_API_KEY in setup cell.")
else:
    try:
        vocals_path = video_info['vocals_path']
        job_dir = vocals_path.parent
        
        print(f"Source audio: {vocals_path}")
        print(f"File size: {format_file_size(vocals_path.stat().st_size)}")
        print(f"Model: whisper-1")
        print(f"Response format: verbose_json")
        print()
        
        # Call OpenAI Whisper API
        print("Calling OpenAI Whisper API...")
        print("-" * 40)
        
        api_start = time.time()
        
        with open(vocals_path, "rb") as audio_file:
            transcript = openai_client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                response_format="verbose_json",
                timestamp_granularities=["segment"]
            )
        
        api_time = time.time() - api_start
        
        print("-" * 40)
        print(f"API call completed in {api_time:.1f} seconds")
        print()
        
        # Extract metadata from response
        detected_language = getattr(transcript, 'language', 'unknown')
        audio_duration = getattr(transcript, 'duration', 0.0)
        full_text = getattr(transcript, 'text', '')
        
        # Parse segments from response
        # verbose_json response has: text, language, duration, segments[]
        # Each segment has: id, seek, start, end, text, tokens, temperature, etc.
        raw_segments = getattr(transcript, 'segments', []) or []
        
        for seg in raw_segments:
            # Handle both dict and object access patterns
            if isinstance(seg, dict):
                start = seg.get('start', 0.0)
                end = seg.get('end', 0.0)
                text = seg.get('text', '').strip()
            else:
                start = getattr(seg, 'start', 0.0)
                end = getattr(seg, 'end', 0.0)
                text = getattr(seg, 'text', '').strip()
            
            transcription_segments.append({
                'start': start,
                'end': end,
                'text': text,
                'duration': end - start
            })
        
        # Store in video_info for later cells
        video_info['transcription_segments'] = transcription_segments
        video_info['detected_language'] = detected_language
        video_info['full_transcript'] = full_text
        
        # Display summary
        print("Transcription Summary")
        print("-" * 40)
        print(f"Detected language: {detected_language}")
        print(f"Audio duration: {audio_duration:.2f} seconds")
        print(f"Total segments: {len(transcription_segments)}")
        print(f"Total characters: {len(full_text)}")
        print()
        
        # Display full transcript preview
        print("Full Transcript (preview)")
        print("-" * 40)
        preview = full_text[:500] + "..." if len(full_text) > 500 else full_text
        print(preview)
        print()
        
        # Display segment table
        print("Transcription Segments")
        print("-" * 80)
        print(f"{'#':<4} {'Start':>8} {'End':>8} {'Text':<56}")
        print("-" * 80)
        
        for i, seg in enumerate(transcription_segments):
            start_str = f"{seg['start']:.2f}s"
            end_str = f"{seg['end']:.2f}s"
            # Truncate text for display
            text_display = seg['text'][:53] + "..." if len(seg['text']) > 53 else seg['text']
            print(f"{i:<4} {start_str:>8} {end_str:>8} {text_display:<56}")
        
        print("-" * 80)
        print()
        
        # Timing summary
        print("Timing")
        print("-" * 40)
        print(f"API call: {api_time:.1f} seconds")
        if audio_duration > 0:
            print(f"Processing ratio: {api_time/audio_duration:.2f}x real-time")
        
        print()
        print("=" * 60)
        print("Transcription complete! Proceed to translation.")
        print("=" * 60)
    
    except Exception as e:
        print(f"ERROR: Transcription failed")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid OPENAI_API_KEY")
        print("  - Audio file too large (max 25MB)")
        print("  - Unsupported audio format")
        print("  - API rate limiting")

## Cell 10: TranslationThis cell translates the transcribed segments using Replicate's **Llama 3.1 405B** model. The LLM approach provides context-aware translation that maintains consistency across segments.### Why Use an LLM for Translation?Traditional machine translation (like Google Translate) works on individual sentences without context. LLM-based translation offers:- **Context awareness**: The model sees surrounding segments, maintaining narrative flow- **Consistent terminology**: Technical terms and names are translated consistently- **Natural phrasing**: Translations sound more natural and conversational- **Tone preservation**: The speaker's style and emotion are maintained### Batching StrategyFor long videos, translation is done in batches to stay within the LLM's context limits:| Parameter | Value | Description ||-----------|-------|-------------|| `LLM_MAX_SEGMENTS_PER_BATCH` | 50 | Maximum segments per LLM call || `LLM_BATCH_OVERLAP` | 5 | Overlapping segments for context continuity || `LLM_MAX_TOKENS` | 4096 | Maximum response tokens || `LLM_TEMPERATURE` | 0.3 | Lower temperature for consistent translations |### Prompt FormatSegments are sent as numbered lines:```0: First segment text1: Second segment text...```The LLM returns translations in the same numbered format, which are then parsed and mapped back to segments.### Output- **translated_segments**: List of segments with both original and translated text- **Comparison table**: Side-by-side display of original vs. translated text- **Batch count**: Number of LLM calls made (for long videos)- **API timing**: Processing time for translation

In [ ]:
# =============================================================================
# Translation - Translate segments using Replicate LLM
# =============================================================================


def translate_batch_llm(transcript_lines: list[str], target_lang_name: str) -> dict[int, str]:
    """
    Translate a batch of transcript lines using Replicate LLM.

    Args:
        transcript_lines: List of "INDEX: text" formatted lines
        target_lang_name: Human-readable language name (e.g., "Spanish")

    Returns:
        Dict mapping segment index to translated text
    """
    transcript = "\n".join(transcript_lines)

    prompt = f"""Translate the following numbered transcript lines to {target_lang_name}.

IMPORTANT RULES:
1. Maintain the exact same line numbers in your output
2. Only output the translated text, preserving the "NUMBER: text" format
3. Keep translations natural and conversational
4. Maintain consistent terminology throughout
5. Preserve the speaker's tone and style
6. Do not add any explanations or notes

Transcript:
{transcript}

Translate each line to {target_lang_name}, keeping the same numbered format:"""

    output = replicate.run(
        REPLICATE_MODEL_LLAMA,
        input={
            "prompt": prompt,
            "max_tokens": LLM_MAX_TOKENS,
            "temperature": LLM_TEMPERATURE,
        }
    )

    # Collect streaming output
    if hasattr(output, '__iter__') and not isinstance(output, str):
        response_text = "".join(str(chunk) for chunk in output)
    else:
        response_text = str(output)

    # Parse the numbered response back into translations
    translations = {}
    for line in response_text.strip().split("\n"):
        line = line.strip()
        if not line or ":" not in line:
            continue

        parts = line.split(":", 1)
        try:
            # Handle potential formatting like "0:" or " 0 :" etc.
            idx_str = parts[0].strip()
            idx = int(idx_str)
            translations[idx] = parts[1].strip()
        except ValueError:
            continue

    return translations


# =============================================================================
# Translation
# =============================================================================

translated_segments = []  # Will hold the translated segments

print("=" * 60)
print("TRANSLATION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('transcription_segments') is None:
    print("ERROR: transcription_segments not available. Run the Transcription cell first.")
else:
    try:
        transcription_segments = video_info['transcription_segments']
        target_lang_name = LANG_NAMES.get(TARGET_LANGUAGE, TARGET_LANGUAGE)
        
        print(f"Source segments: {len(transcription_segments)}")
        print(f"Target language: {TARGET_LANGUAGE} ({target_lang_name})")
        print(f"Model: {REPLICATE_MODEL_LLAMA}")
        print(f"Batch size: {LLM_MAX_SEGMENTS_PER_BATCH} segments (overlap: {LLM_BATCH_OVERLAP})")
        print()
        
        if not transcription_segments:
            print("WARNING: No transcription segments to translate.")
            translated_segments = []
        else:
            # Build numbered transcript for the LLM
            transcript_lines = []
            segment_indices = []  # Track which segments have text
            for i, seg in enumerate(transcription_segments):
                text = seg.get("text", "").strip()
                if text:
                    transcript_lines.append(f"{i}: {text}")
                    segment_indices.append(i)
            
            if not transcript_lines:
                print("WARNING: No text content in segments to translate.")
                translated_segments = [{
                    "start": s.get("start", 0.0),
                    "end": s.get("end", 0.0),
                    "duration": s.get("duration", 0.0),
                    "original_text": s.get("text", ""),
                    "translated_text": ""
                } for s in transcription_segments]
            else:
                print(f"Segments with text: {len(transcript_lines)}")
                print()
                
                # Determine batching strategy
                all_translations = {}
                batch_count = 0
                
                if len(transcript_lines) <= LLM_MAX_SEGMENTS_PER_BATCH:
                    # Single batch - translate all at once
                    print("Translating all segments in a single batch...")
                    print("-" * 40)
                    
                    api_start = time.time()
                    batch_translations = translate_batch_llm(transcript_lines, target_lang_name)
                    api_time = time.time() - api_start
                    
                    all_translations = batch_translations
                    batch_count = 1
                    
                    print("-" * 40)
                    print(f"Batch completed in {api_time:.1f} seconds")
                    print(f"Translated {len(batch_translations)}/{len(transcript_lines)} segments")
                else:
                    # Multiple batches with overlap for context continuity
                    total_batches = (len(transcript_lines) + LLM_MAX_SEGMENTS_PER_BATCH - LLM_BATCH_OVERLAP - 1) // (LLM_MAX_SEGMENTS_PER_BATCH - LLM_BATCH_OVERLAP)
                    print(f"Translating {len(transcript_lines)} segments in {total_batches} batches...")
                    print("-" * 40)
                    
                    api_start = time.time()
                    
                    for batch_start in range(0, len(transcript_lines), LLM_MAX_SEGMENTS_PER_BATCH - LLM_BATCH_OVERLAP):
                        batch_end = min(batch_start + LLM_MAX_SEGMENTS_PER_BATCH, len(transcript_lines))
                        batch_lines = transcript_lines[batch_start:batch_end]
                        
                        batch_count += 1
                        print(f"  Batch {batch_count}/{total_batches}: segments {batch_start}-{batch_end-1}")
                        
                        batch_translations = translate_batch_llm(batch_lines, target_lang_name)
                        
                        # Merge translations (later batches overwrite overlap regions)
                        all_translations.update(batch_translations)
                        print(f"    -> Translated {len(batch_translations)} segments")
                    
                    api_time = time.time() - api_start
                    print("-" * 40)
                    print(f"All batches completed in {api_time:.1f} seconds")
                
                print()
                
                # Build translated segments list
                for i, seg in enumerate(transcription_segments):
                    original_text = seg.get("text", "")
                    translated_text = all_translations.get(i, original_text)  # Fallback to original
                    
                    translated_segments.append({
                        "start": seg.get("start", 0.0),
                        "end": seg.get("end", 0.0),
                        "duration": seg.get("duration", 0.0),
                        "original_text": original_text,
                        "translated_text": translated_text
                    })
                
                # Store in video_info for later cells
                video_info['translated_segments'] = translated_segments
                
                # Display summary
                print("Translation Summary")
                print("-" * 40)
                print(f"Total segments: {len(translated_segments)}")
                print(f"Segments translated: {len(all_translations)}")
                print(f"Batches used: {batch_count}")
                print()
                
                # Display translation comparison table
                print("Translation Comparison")
                print("-" * 100)
                print(f"{'#':<4} {'Original Text':<45} {'Translated Text':<45}")
                print("-" * 100)
                
                for i, seg in enumerate(translated_segments):
                    orig = seg['original_text']
                    trans = seg['translated_text']
                    # Truncate for display
                    orig_display = orig[:42] + "..." if len(orig) > 42 else orig
                    trans_display = trans[:42] + "..." if len(trans) > 42 else trans
                    print(f"{i:<4} {orig_display:<45} {trans_display:<45}")
                
                print("-" * 100)
                print()
                
                # Timing summary
                print("Timing")
                print("-" * 40)
                print(f"API calls: {batch_count}")
                print(f"Total time: {api_time:.1f} seconds")
                if batch_count > 0:
                    print(f"Average per batch: {api_time/batch_count:.1f} seconds")
        
        print()
        print("=" * 60)
        print("Translation complete! Proceed to speech synthesis.")
        print("=" * 60)
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate API call failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Model unavailable")
    
    except Exception as e:
        print(f"ERROR: Translation failed")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")

## Cell 11: Single-Speaker Synthesis

This cell synthesizes all translated text using a **single voice**. Use this mode when `USE_MULTI_SPEAKER = False` or as a fallback when speaker diarization is unavailable.

### When to Use Single-Speaker Mode

Single-speaker synthesis is ideal for:

- **Single-speaker content**: Tutorials, narration, monologues
- **Faster processing**: No per-segment synthesis, just one API call
- **Fallback mode**: When voice samples couldn't be extracted for all speakers

### Chatterbox API Details

The Replicate `chatterbox-multilingual` model accepts:

| Parameter | Description |
|-----------|-------------|
| `text` | The text to synthesize (all translated segments combined) |
| `language` | Target language code (e.g., "es", "ja") |
| `reference_audio` | URL to voice sample for cloning |
| `cfg_weight` | CFG/Pace weight (0.0-1.0, higher = slower/more stable) |
| `exaggeration` | Voice exaggeration (0.0-1.0, higher = more expressive) |

### Voice Sample Selection

In single-speaker mode, we use the **first available voice sample** from the voice sample extraction step. If no samples were extracted, we fall back to using the original vocals file.

### Output

- **translated_audio_single.wav**: The synthesized audio file at Chatterbox's native 24kHz sample rate
- **Audio duration**: Duration of the generated speech (may differ from original due to speech rate)
- **API timing**: Processing time for synthesis

In [ ]:
# =============================================================================
# Single-Speaker Synthesis - Synthesize all text with one voice
# =============================================================================

# =============================================================================
# Single-Speaker Synthesis
# =============================================================================

single_speaker_output = None  # Will hold the path to synthesized audio

print("=" * 60)
print("SINGLE-SPEAKER SYNTHESIS")
print("=" * 60)
print()

# Check if this mode should run
if USE_MULTI_SPEAKER:
    print("Skipping: USE_MULTI_SPEAKER is True")
    print("  This cell only runs when USE_MULTI_SPEAKER is False.")
    print("  Proceed to the Multi-Speaker Synthesis cell instead.")
# Check prerequisites from previous cells
elif video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('translated_segments') is None:
    print("ERROR: translated_segments not available. Run the Translation cell first.")
else:
    try:
        translated_segments = video_info['translated_segments']
        job_dir = video_info['video_path'].parent
        
        # Combine all translated text
        combined_text = " ".join(
            seg.get('translated_text', '').strip()
            for seg in translated_segments
            if seg.get('translated_text', '').strip()
        )
        
        if not combined_text:
            print("ERROR: No translated text available for synthesis.")
        else:
            print(f"Text to synthesize: {len(combined_text)} characters")
            print(f"  Preview: {combined_text[:100]}..." if len(combined_text) > 100 else f"  Text: {combined_text}")
            print()
            
            # Get voice sample URL - use first available or fall back to vocals
            speaker_voice_urls = video_info.get('speaker_voice_urls', {})
            vocals_path = video_info.get('vocals_path')
            
            voice_sample_url = None
            voice_source = None
            
            if speaker_voice_urls:
                # Use first available voice sample
                first_speaker = sorted(speaker_voice_urls.keys())[0]
                voice_sample_url = speaker_voice_urls[first_speaker]
                voice_source = f"voice sample from {first_speaker}"
                print(f"Using voice sample: {first_speaker}")
            elif vocals_path and vocals_path.exists():
                # Upload vocals as fallback voice sample
                print("No voice samples available, uploading vocals as reference...")
                upload_start = time.time()
                uploaded_file = replicate.files.create(file=vocals_path)
                voice_sample_url = uploaded_file.urls['get']
                voice_source = "original vocals (fallback)"
                print(f"  Upload completed in {time.time() - upload_start:.1f}s")
            else:
                raise ValueError("No voice sample or vocals available for synthesis")
            
            print(f"Voice source: {voice_source}")
            print()
            
            # Configure Chatterbox parameters
            cfg_weight = CHATTERBOX_CFG_WEIGHT
            exaggeration = CHATTERBOX_EXAGGERATION
            
            print(f"Model: {REPLICATE_MODEL_CHATTERBOX.split(':')[0]}")
            print(f"Target language: {TARGET_LANGUAGE}")
            print(f"CFG weight: {cfg_weight}")
            print(f"Exaggeration: {exaggeration}")
            print()
            
            # Call Replicate Chatterbox API
            print("Calling Replicate Chatterbox API...")
            print("-" * 40)
            
            api_start = time.time()
            
            output = replicate.run(
                REPLICATE_MODEL_CHATTERBOX,
                input={
                    "text": combined_text,
                    "language": TARGET_LANGUAGE,
                    "reference_audio": voice_sample_url,
                    "cfg_weight": cfg_weight,
                    "exaggeration": exaggeration,
                }
            )
            
            api_time = time.time() - api_start
            
            print("-" * 40)
            print(f"API call completed in {api_time:.1f} seconds")
            print()
            
            # Download and save the output audio
            single_speaker_output = job_dir / "translated_audio_single.wav"
            
            print("Downloading synthesized audio...")
            download_start = time.time()
            
            # Handle different output types from Replicate
            if isinstance(output, FileOutput):
                # Replicate v1.0+ returns FileOutput objects
                with open(single_speaker_output, "wb") as f:
                    for chunk in output:
                        f.write(chunk)
            elif isinstance(output, str):
                # Output is a URL - download with httpx
                with httpx.Client(timeout=None) as client:
                    response = client.get(output)
                    single_speaker_output.write_bytes(response.content)
            else:
                # Output might be file-like or iterator
                with open(single_speaker_output, "wb") as f:
                    for chunk in output:
                        f.write(chunk)
            
            download_time = time.time() - download_start
            print(f"Download completed in {download_time:.1f} seconds")
            print()
            
            # Get output file info
            output_size = single_speaker_output.stat().st_size
            output_duration = get_audio_duration(single_speaker_output)
            
            # Store in video_info for later cells
            video_info['single_speaker_output'] = single_speaker_output
            video_info['synthesized_audio_path'] = single_speaker_output  # Generic reference
            
            # Display summary
            print("Single-Speaker Synthesis Summary")
            print("-" * 40)
            print(f"Output file: {single_speaker_output}")
            print(f"File size: {format_file_size(output_size)}")
            print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f} seconds)")
            print(f"Sample rate: {CHATTERBOX_SAMPLE_RATE} Hz (Chatterbox native)")
            print()
            
            # Timing summary
            print("Timing")
            print("-" * 40)
            print(f"API call: {api_time:.1f} seconds")
            print(f"Download: {download_time:.1f} seconds")
            print(f"Total: {api_time + download_time:.1f} seconds")
            
            # Calculate real-time factor
            if output_duration > 0:
                rtf = (api_time + download_time) / output_duration
                print(f"Real-time factor: {rtf:.2f}x")
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate API call failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Text too long for model")
        print("  - Invalid voice sample format")
    
    except httpx.HTTPError as e:
        print(f"ERROR: Failed to download synthesized audio")
        print(f"  Details: {str(e)}")
    
    except Exception as e:
        print(f"ERROR: Single-speaker synthesis failed")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")

print()
print("=" * 60)
if single_speaker_output and single_speaker_output.exists():
    print("Single-speaker synthesis complete! Proceed to audio mixing.")
elif USE_MULTI_SPEAKER:
    print("Single-speaker mode skipped. Proceed to multi-speaker synthesis.")
else:
    print("Single-speaker synthesis failed. Check errors above.")
print("=" * 60)

## Cell 12: Multi-Speaker Synthesis

This cell synthesizes each translated segment using the **matched speaker's voice**. Use this mode when `USE_MULTI_SPEAKER = True` and speaker diarization has successfully identified multiple speakers.

### When to Use Multi-Speaker Mode

Multi-speaker synthesis is ideal for:

- **Multi-speaker content**: Interviews, conversations, podcasts, panel discussions
- **Voice preservation**: Maintains distinct voices for each speaker
- **Natural flow**: Each segment matches the original speaker's voice characteristics

### Speaker Matching Algorithm

Each transcription segment is matched to a speaker by finding the **maximum overlap** between:
- The segment's time range (from Whisper transcription)
- The diarization segments (from speaker-diarization model)

```python
def match_segment_to_speaker(seg_start, seg_end):
    for diar_seg in diarization_segments:
        overlap_start = max(seg_start, diar_seg['start'])
        overlap_end = min(seg_end, diar_seg['end'])
        overlap = max(0, overlap_end - overlap_start)
        # Track speaker with maximum overlap
```

### Timing Adjustment

To prevent audio overlap between segments:
1. Each segment starts at `max(original_start_time, previous_segment_end_time)`
2. If a segment hasn't finished playing, the next segment waits
3. This preserves timing relationships while avoiding audio collision

### Chatterbox API (Per-Segment)

For each segment, we call the Chatterbox API with:

| Parameter | Description |
|-----------|-------------|
| `text` | The translated text for this segment |
| `language` | Target language code |
| `reference_audio` | Voice sample URL for the matched speaker |
| `cfg_weight` | CFG/Pace weight (0.0-1.0) |
| `exaggeration` | Voice exaggeration (0.0-1.0) |

### Output

- **translated_audio_multi.wav**: Combined audio with all segments at adjusted timing
- **Segment-to-speaker mapping table**: Shows which speaker was assigned to each segment
- **Per-segment progress**: Real-time feedback during synthesis
- **API timing**: Total processing time for all segments

In [ ]:
# =============================================================================
# Multi-Speaker Synthesis - Synthesize each segment with matched speaker voice
# =============================================================================

import numpy as np

# =============================================================================
# Multi-Speaker Synthesis
# =============================================================================

multi_speaker_output = None  # Will hold the path to synthesized audio

print("=" * 60)
print("MULTI-SPEAKER SYNTHESIS")
print("=" * 60)
print()

# Check if this mode should run
if not USE_MULTI_SPEAKER:
    print("Skipping: USE_MULTI_SPEAKER is False")
    print("  This cell only runs when USE_MULTI_SPEAKER is True.")
    print("  The single-speaker synthesis cell should have run instead.")
# Check prerequisites from previous cells
elif video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('translated_segments') is None:
    print("ERROR: translated_segments not available. Run the Translation cell first.")
elif video_info.get('diarization_segments') is None:
    print("ERROR: diarization_segments not available. Run the Speaker Diarization cell first.")
elif video_info.get('speaker_voice_urls') is None or len(video_info.get('speaker_voice_urls', {})) == 0:
    print("ERROR: speaker_voice_urls not available or empty. Run the Voice Sample Extraction cell first.")
    print("  Falling back to single-speaker mode may be necessary.")
else:
    try:
        translated_segments = video_info['translated_segments']
        diarization_segments = video_info['diarization_segments']
        speaker_voice_urls = video_info['speaker_voice_urls']
        job_dir = video_info['video_path'].parent
        
        print(f"Translated segments: {len(translated_segments)}")
        print(f"Diarization segments: {len(diarization_segments)}")
        print(f"Speaker voice URLs: {len(speaker_voice_urls)}")
        print(f"  Speakers: {', '.join(sorted(speaker_voice_urls.keys()))}")
        print()
        
        # Configure Chatterbox parameters
        cfg_weight = CHATTERBOX_CFG_WEIGHT
        exaggeration = CHATTERBOX_EXAGGERATION
        
        print(f"Model: {REPLICATE_MODEL_CHATTERBOX.split(':')[0]}")
        print(f"Target language: {TARGET_LANGUAGE}")
        print(f"CFG weight: {cfg_weight}")
        print(f"Exaggeration: {exaggeration}")
        print()
        
        # Get fallback voice URL (first available speaker)
        fallback_speaker = sorted(speaker_voice_urls.keys())[0]
        fallback_voice_url = speaker_voice_urls[fallback_speaker]
        
        def match_segment_to_speaker(seg_start: float, seg_end: float) -> str:
            """Find the speaker with maximum overlap for a given segment time range."""
            best_speaker = fallback_speaker
            best_overlap = 0.0
            
            for diar_seg in diarization_segments:
                # Calculate overlap between transcription segment and diarization segment
                overlap_start = max(seg_start, diar_seg['start'])
                overlap_end = min(seg_end, diar_seg['end'])
                overlap = max(0.0, overlap_end - overlap_start)
                
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_speaker = diar_seg['speaker']
            
            return best_speaker
        
        # Process each segment and collect audio with original start times
        total_segments = len(translated_segments)
        segment_audios: list[tuple[float, np.ndarray]] = []  # (original_start, audio)
        segment_speakers = []  # Track speaker assignments for display
        
        # Target sample rate for Chatterbox output
        target_sample_rate = CHATTERBOX_SAMPLE_RATE
        
        print("Processing segments...")
        print("-" * 60)
        
        total_api_start = time.time()
        
        for i, seg in enumerate(translated_segments):
            text = seg.get('translated_text', '').strip()
            
            if not text:
                # Skip empty segments
                segment_speakers.append((i, seg['start'], seg['end'], '(skipped)', '(empty)'))
                continue
            
            # Match this segment to a speaker
            speaker = match_segment_to_speaker(seg['start'], seg['end'])
            voice_url = speaker_voice_urls.get(speaker, fallback_voice_url)
            
            segment_speakers.append((i, seg['start'], seg['end'], speaker, text[:30] + '...' if len(text) > 30 else text))
            
            # Display progress
            print(f"Segment {i+1}/{total_segments}: {speaker} ({seg['start']:.1f}s - {seg['end']:.1f}s)")
            
            # Call Replicate Chatterbox API for this segment
            try:
                segment_start = time.time()
                
                output = replicate.run(
                    REPLICATE_MODEL_CHATTERBOX,
                    input={
                        "text": text,
                        "language": TARGET_LANGUAGE,
                        "reference_audio": voice_url,
                        "cfg_weight": cfg_weight,
                        "exaggeration": exaggeration,
                    }
                )
                
                # Download the segment audio
                segment_audio_path = job_dir / f"segment_{i}.wav"
                
                if isinstance(output, FileOutput):
                    # Replicate v1.0+ returns FileOutput objects
                    with open(segment_audio_path, "wb") as f:
                        for chunk in output:
                            f.write(chunk)
                elif isinstance(output, str):
                    # Output is a URL
                    with httpx.Client(timeout=None) as client:
                        response = client.get(output)
                        segment_audio_path.write_bytes(response.content)
                else:
                    # Output might be file-like or iterator
                    with open(segment_audio_path, "wb") as f:
                        for chunk in output:
                            f.write(chunk)
                
                # Convert to 16-bit PCM (Chatterbox outputs floating-point WAV)
                converted_path = job_dir / f"segment_{i}_converted.wav"
                subprocess.run([
                    "ffmpeg", "-y",
                    "-i", str(segment_audio_path),
                    "-acodec", "pcm_s16le",
                    str(converted_path)
                ], capture_output=True, check=True)
                
                # Read the converted audio data
                sr, audio_np = read_wav_file(converted_path)
                
                # Resample if needed
                if sr != target_sample_rate:
                    audio_np = resample_audio(audio_np, sr, target_sample_rate)
                
                segment_audios.append((seg['start'], audio_np))
                
                # Clean up segment files
                segment_audio_path.unlink(missing_ok=True)
                converted_path.unlink(missing_ok=True)
                
                segment_time = time.time() - segment_start
                print(f"  -> Completed in {segment_time:.1f}s")
                
            except Exception as e:
                print(f"  -> FAILED: {str(e)[:50]}")
                continue
        
        print("-" * 60)
        total_api_time = time.time() - total_api_start
        print(f"All segments processed in {total_api_time:.1f} seconds")
        print()
        
        if not segment_audios:
            print("ERROR: No segments were successfully synthesized!")
        else:
            # Combine all segments with adjusted timing to prevent overlap
            print("Combining segments with adjusted timing...")
            
            positioned_audios: list[tuple[float, np.ndarray]] = []
            current_end_time = 0.0
            
            for original_start, audio in segment_audios:
                audio_duration = len(audio) / target_sample_rate
                
                # Start at the later of: original timestamp or when previous segment ends
                actual_start = max(original_start, current_end_time)
                positioned_audios.append((actual_start, audio))
                
                # Update end time for next segment
                current_end_time = actual_start + audio_duration
            
            # Calculate output length
            total_duration = max(
                translated_segments[-1].get('end', 0.0) if translated_segments else 0.0,
                current_end_time
            )
            output_length = int(total_duration * target_sample_rate)
            output_audio = np.zeros(output_length)
            
            # Place audio at calculated positions
            for start_time, audio in positioned_audios:
                start_sample = int(start_time * target_sample_rate)
                end_sample = start_sample + len(audio)
                
                # Ensure we don't overflow
                if end_sample > output_length:
                    audio = audio[:output_length - start_sample]
                
                # Place audio
                if start_sample < output_length:
                    output_audio[start_sample:start_sample + len(audio)] = audio
            
            # Normalize to prevent clipping
            max_val = np.max(np.abs(output_audio))
            if max_val > 0.99:
                output_audio = output_audio * 0.99 / max_val
            
            # Save the combined audio
            multi_speaker_output = job_dir / "translated_audio_multi.wav"
            write_wav_file(multi_speaker_output, target_sample_rate, output_audio)
            
            # Store in video_info for later cells
            video_info['multi_speaker_output'] = multi_speaker_output
            video_info['synthesized_audio_path'] = multi_speaker_output  # Generic reference
            
            # Get output file info
            output_size = multi_speaker_output.stat().st_size
            output_duration = get_audio_duration(multi_speaker_output)
            
            print()
            print("Segment-to-Speaker Mapping")
            print("-" * 100)
            print(f"{'#':<4} {'Start':<8} {'End':<8} {'Speaker':<12} {'Text Preview':<60}")
            print("-" * 100)
            for idx, start, end, speaker, text in segment_speakers:
                print(f"{idx:<4} {start:<8.1f} {end:<8.1f} {speaker:<12} {text:<60}")
            print("-" * 100)
            print()
            
            # Display summary
            print("Multi-Speaker Synthesis Summary")
            print("-" * 40)
            print(f"Output file: {multi_speaker_output}")
            print(f"File size: {format_file_size(output_size)}")
            print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f} seconds)")
            print(f"Sample rate: {CHATTERBOX_SAMPLE_RATE} Hz (Chatterbox native)")
            print(f"Segments synthesized: {len(segment_audios)}/{total_segments}")
            print()
            
            # Timing summary
            print("Timing")
            print("-" * 40)
            print(f"Total API time: {total_api_time:.1f} seconds")
            if len(segment_audios) > 0:
                print(f"Average per segment: {total_api_time/len(segment_audios):.1f} seconds")
            if output_duration > 0:
                rtf = total_api_time / output_duration
                print(f"Real-time factor: {rtf:.2f}x")
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate API call failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Text too long for model")
        print("  - Invalid voice sample format")
    
    except httpx.HTTPError as e:
        print(f"ERROR: Failed to download synthesized audio")
        print(f"  Details: {str(e)}")
    
    except Exception as e:
        print(f"ERROR: Multi-speaker synthesis failed")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")

print()
print("=" * 60)
if multi_speaker_output and multi_speaker_output.exists():
    print("Multi-speaker synthesis complete! Proceed to audio mixing.")
elif not USE_MULTI_SPEAKER:
    print("Multi-speaker mode skipped. Single-speaker synthesis should be used.")
else:
    print("Multi-speaker synthesis failed. Check errors above.")
print("=" * 60)

## Cell 13: Audio Mixing

This cell mixes the translated speech with the original background audio extracted during the separation step. This creates a natural-sounding output where the translated speech plays over the original music/effects.

### Mixing Process

The mixing uses **ffmpeg's amix filter** to combine two audio tracks:

1. **Speech track** (100% volume): The synthesized translated speech
2. **Background track** (configurable volume): Original background audio from Demucs separation

```
ffmpeg -i speech.wav -i background.wav \
  -filter_complex '[1:a]apad,volume={vol}[bg];[0:a][bg]amix=inputs=2:duration=first:dropout_transition=0' \
  -ac 2 -y mixed_audio.wav
```

### Filter Breakdown

| Component | Description |
|-----------|-------------|
| `[1:a]apad` | Pad background audio with silence if shorter than speech |
| `volume={vol}` | Apply volume adjustment to background (0.0 to 1.0) |
| `[bg]` | Label for processed background stream |
| `amix=inputs=2` | Mix two audio streams together |
| `duration=first` | Output duration matches the speech track (first input) |
| `dropout_transition=0` | No fade when one input ends |
| `-ac 2` | Output stereo audio |

### Configuration

| Variable | Default | Description |
|----------|---------|-------------|
| `BACKGROUND_VOLUME` | 0.3 | Volume level for background (0.0 = muted, 1.0 = full) |

Typical values:
- **0.1 - 0.2**: Background barely audible, speech-focused
- **0.3** (default): Balanced mix, background clearly present
- **0.4 - 0.5**: Background more prominent, cinematic feel
- **0.0**: Background muted (speech only)

### Output

- **mixed_audio.wav**: Final mixed audio with speech and background
- Duration matches the synthesized speech track
- Stereo output format


In [ ]:
# =============================================================================
# Audio Mixing - Mix translated speech with background audio
# =============================================================================

# =============================================================================
# Configuration
# =============================================================================

# Background volume level (0.0 to 1.0)
# - 0.3 (30%) is the default, providing a balanced mix
# - Adjust this to control how prominent the background music/effects are
BACKGROUND_VOLUME = BACKGROUND_VOLUME_DEFAULT  # From config.py (0.3)

# =============================================================================
# Audio Mixing
# =============================================================================

mixed_audio_path = None  # Will hold the path to mixed audio

print("=" * 60)
print("AUDIO MIXING")
print("=" * 60)
print()

# Check prerequisites
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('synthesized_audio_path') is None:
    print("ERROR: synthesized_audio_path not available.")
    print("  Run either the Single-Speaker or Multi-Speaker Synthesis cell first.")
elif video_info.get('background_path') is None:
    print("ERROR: background_path not available. Run the Audio Separation cell first.")
elif not video_info['synthesized_audio_path'].exists():
    print(f"ERROR: Synthesized audio file not found: {video_info['synthesized_audio_path']}")
elif not video_info['background_path'].exists():
    print(f"ERROR: Background audio file not found: {video_info['background_path']}")
else:
    try:
        speech_path = video_info['synthesized_audio_path']
        background_path = video_info['background_path']
        job_dir = video_info['video_path'].parent
        
        print(f"Speech audio: {speech_path.name}")
        print(f"  Size: {format_file_size(speech_path.stat().st_size)}")
        speech_duration = get_audio_duration(speech_path)
        print(f"  Duration: {format_duration(int(speech_duration))} ({speech_duration:.2f}s)")
        print()
        print(f"Background audio: {background_path.name}")
        print(f"  Size: {format_file_size(background_path.stat().st_size)}")
        background_duration = get_audio_duration(background_path)
        print(f"  Duration: {format_duration(int(background_duration))} ({background_duration:.2f}s)")
        print()
        
        # Validate and clamp background volume
        bg_volume = max(0.0, min(1.0, BACKGROUND_VOLUME))
        if bg_volume != BACKGROUND_VOLUME:
            print(f"WARNING: BACKGROUND_VOLUME clamped from {BACKGROUND_VOLUME} to {bg_volume}")
        
        print(f"Background volume: {bg_volume:.0%}")
        print()
        
        # Output path
        mixed_audio_path = job_dir / "mixed_audio.wav"
        
        print("Mixing audio with ffmpeg...")
        mix_start = time.time()
        
        # Build the ffmpeg filter for mixing
        # [1:a]apad: Pad background with silence if shorter than speech
        # volume={vol}: Apply volume to background
        # amix: Mix the two streams
        # duration=first: Output duration matches the speech (first input)
        filter_complex = (
            f'[1:a]apad,volume={bg_volume}[bg];'
            f'[0:a][bg]amix=inputs=2:duration=first:dropout_transition=0'
        )
        
        result = subprocess.run([
            'ffmpeg',
            '-i', str(speech_path),
            '-i', str(background_path),
            '-filter_complex', filter_complex,
            '-ac', '2',  # Stereo output
            '-y',  # Overwrite output
            str(mixed_audio_path)
        ], capture_output=True, text=True)
        
        mix_time = time.time() - mix_start
        
        if result.returncode != 0:
            print(f"ERROR: ffmpeg mixing failed")
            print(f"  stderr: {result.stderr[:500]}..." if len(result.stderr) > 500 else f"  stderr: {result.stderr}")
            mixed_audio_path = None
        else:
            # Verify output exists
            if not mixed_audio_path.exists():
                print("ERROR: Output file was not created")
                mixed_audio_path = None
            else:
                # Get output info
                output_size = mixed_audio_path.stat().st_size
                output_duration = get_audio_duration(mixed_audio_path)
                
                # Store in video_info for later cells
                video_info['mixed_audio_path'] = mixed_audio_path
                
                print()
                print("Audio Mixing Summary")
                print("-" * 40)
                print(f"Output file: {mixed_audio_path}")
                print(f"File size: {format_file_size(output_size)}")
                print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f} seconds)")
                print(f"Background volume: {bg_volume:.0%}")
                print(f"Processing time: {mix_time:.2f} seconds")
                print()
                
                # Compare durations
                if abs(output_duration - speech_duration) > 0.5:
                    print(f"NOTE: Output duration differs from speech by {abs(output_duration - speech_duration):.2f}s")
    
    except subprocess.CalledProcessError as e:
        print(f"ERROR: ffmpeg command failed")
        print(f"  Return code: {e.returncode}")
        if e.stderr:
            print(f"  stderr: {e.stderr[:500]}..." if len(str(e.stderr)) > 500 else f"  stderr: {e.stderr}")
        mixed_audio_path = None
    
    except Exception as e:
        print(f"ERROR: Audio mixing failed")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")
        mixed_audio_path = None

print()
print("=" * 60)
if mixed_audio_path and mixed_audio_path.exists():
    print(f"Audio mixing complete! Output saved to: {mixed_audio_path}")
    print("Proceed to Subtitle Generation cell.")
else:
    print("Audio mixing failed. Check errors above.")
print("=" * 60)
